Snowflake data loading concepts and RESULT_SCAN reference notebook

*Co-authored with CoCo*

# Snowflake: Loading Data into Tables

## Two Main Methods of Loading Data

### 1. Bulk Loading
- **Most frequent and most important** method for loading data into Snowflake
- Uses the **compute power of your own warehouses** (your own capacity is consumed)
- Involves **Stages** — intermediate locations where data is loaded from
- Uses the **COPY command** — a secure command to load data from stages into tables
- **Allows data transformations** during load (e.g., cleaning data, minor transformations)
- Typically used for **higher volume** of data loaded periodically

### 2. Continuous Loading
- Used when data needs to be **available immediately / up to date**
- Typically involves **smaller amounts** of data compared to bulk loading
- Uses **Serverless features** — managed by Snowflake (not your dedicated warehouse)
- The feature used for continuous loading is **Snowpipe**

## Key Differences Summary

| Aspect | Bulk Loading | Continuous Loading |
|--------|-------------|-------------------|
| Compute | Your own warehouse | Serverless (Snowflake-managed) |
| Volume | Higher volume, periodic | Smaller, near real-time |
| Command/Feature | COPY command | Snowpipe |
| Use Case | Batch/periodic loads | Data needed immediately |

# Snowflake Stages

## What is a Stage?
- **Not to be confused** with the general "staging area" concept in data warehousing
- A **Stage** is a database object in Snowflake
- It contains the **location of data files** from where data can be loaded
- Created using the `CREATE STAGE` command (similar to creating a table)
- Has **properties** associated with it:
  - **URL / Location** (e.g., an AWS S3 bucket path)
  - **Credentials** (access settings for the external location)
  - **Additional connection settings**

## Two Types of Stages

### 1. External Stage (Most Common)
- Refers to an **external cloud storage location**:
  - AWS S3 Bucket
  - Google Cloud Storage
  - Azure Blob Storage
- Created within a **database schema** using `CREATE STAGE`
- It is a **database object** that lives inside your schema
- Includes properties: URL, access settings, connection settings

**Cost Consideration:**
- If the external stage is in a **different region or different cloud platform** than your Snowflake account, **additional data transfer costs** may occur
- Example: Snowflake account on AWS US-East, but stage points to Azure or a different AWS region → extra transfer fees
- **Incoming data to Snowflake** is free, but **outgoing data** from your cloud provider may incur charges
- Always consider potential data transfer costs when the stage location differs from your account region/platform

> **Clarification on "Incoming free, Outgoing costs":**
> When you load data **into** Snowflake (ingress), Snowflake does not charge for that data transfer. However, if your data source is on a different cloud provider or region (e.g., Azure Blob Storage), **your cloud provider** (Azure in this case) may charge you for sending data **out** of their platform (egress fees). The cost isn't from Snowflake's side — it's from the source cloud provider charging for outbound data transfer.

### 2. Internal Stage
- Used when there is **no access to an external cloud provider**
- Example: loading from a local file or on-premise server
- Less frequently used compared to external stages

## Key Takeaway
- **External stages** are the most common and important type
- A stage is essentially a pointer to your data location + all associated connection properties, making it easy to load data into Snowflake

# SQL Commands: External Stages & Loading Data

---

### 1. Create a Database for Managing Stages
```sql
CREATE OR REPLACE DATABASE MANAGE_DB;
```
> A dedicated database to organize stage objects, file formats, and other data loading resources.

---

### 2. Create a Schema for External Stages
```sql
CREATE OR REPLACE SCHEMA external_stages;
```
> Keeps external stage objects organized under a specific schema.

---

### 3. Create an External Stage (with credentials)
```sql
CREATE OR REPLACE STAGE MANAGE_DB.external_stages.aws_stage
    url='s3://bucketsnowflakes3'
    credentials=(aws_key_id='ABCD_DUMMY_ID' aws_secret_key='1234abcd_key');
```
> Points to a private S3 bucket and includes AWS credentials for access.

---

### 4. Describe the Stage
```sql
DESC STAGE MANAGE_DB.external_stages.aws_stage;
```
> View all properties of the stage — URL, credentials, file format settings, etc.

---

### 5. Alter Stage Credentials
```sql
ALTER STAGE aws_stage
    SET credentials=(aws_key_id='XYZ_DUMMY_ID' aws_secret_key='987xyz');
```
> Update the access credentials without recreating the stage.

---

### 6. Create a Publicly Accessible Stage (no credentials)
```sql
CREATE OR REPLACE STAGE MANAGE_DB.external_stages.aws_stage
    url='s3://bucketsnowflakes3';
```
> For public S3 buckets, no credentials are needed.

---

### 7. List Files in the Stage
```sql
LIST @aws_stage;
```
> Shows all files available in the stage location.

---

### 8. Load Data Using COPY Command
```sql
COPY INTO OUR_FIRST_DB.PUBLIC.ORDERS
    FROM @aws_stage
    file_format= (type = csv field_delimiter=',' skip_header=1)
    pattern='.*Order.*';
```
> Loads CSV files matching the pattern `*Order*` from the stage into the `ORDERS` table, skipping the header row.

---

## Note:- When to Use `@` Symbol

**Use `@` — when referencing a stage to access its contents:**
- `LIST @aws_stage;` — listing files inside the stage
- `COPY INTO ... FROM @aws_stage` — loading data from the stage
- `SELECT $1 FROM @aws_stage` — querying files in the stage

**Don't use `@` — when managing the stage object itself (DDL):**
- `CREATE STAGE aws_stage ...` — creating it
- `ALTER STAGE aws_stage ...` — modifying it
- `DESC STAGE aws_stage` — describing it
- `DROP STAGE aws_stage` — deleting it

> **Simple rule:** `@` means "look inside this stage's files." If you're working with the stage as a database object (DDL operations), no `@`. If you're accessing the data/files it points to, use `@`.

# RESULT_SCAN

`RESULT_SCAN` allows you to retrieve the results of a previously executed query in Snowflake as a virtual table, enabling further querying without rerunning the original query.

## Overview

The `RESULT_SCAN` function in Snowflake is a table function that returns the result set of a previous query as if it were a table. This is particularly useful for:

- Inspecting or validating results from prior queries during debugging
- Processing outputs from `SHOW` or `DESCRIBE` commands
- Accessing results from queries on metadata or account usage
- Using results from stored procedures that return tabular data

`RESULT_SCAN` can reference queries from the current session or other sessions, as long as the query was executed within the last 24 hours.

## Syntax and Usage

The basic syntax is:

```sql
SELECT *
FROM TABLE(RESULT_SCAN('<query_id>'));
```

Where `<query_id>` is the unique identifier of the query whose results you want to retrieve. You can obtain the query ID from:

- Snowflake Web UI under **Monitoring > Query History**
- `QUERY_HISTORY` table function
- `LAST_QUERY_ID()` function for the most recent query in the current session

For example, retrieving the last query's results can be done with:

```sql
SELECT *
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
```

You can also apply additional SQL clauses like `WHERE` or `ORDER BY` to filter or sort the results differently from the original query.

## Practical Considerations

- **Duplicate column names** in the original query are automatically renamed with suffixes (`_1`, `_2`, etc.) to ensure uniqueness.
- **Performance:** `RESULT_SCAN` returns cached results, which is faster than rerunning the query, but large result sets may be slower to process than querying a table directly.
- **Integration:** It is useful in ELT workflows, allowing intermediate results to be reused without creating temporary tables, reducing compute costs.

## Example

Suppose you run a query to calculate total sales per region:

```sql
SELECT region, SUM(sales) AS total_sales
FROM sales_data
GROUP BY region;
```

After capturing the query ID (e.g., `01a1b2c3-d4e5-f6g7-h8i9-0j1k2l3m4n5o`), you can retrieve and further filter the results:

```sql
SELECT *
FROM TABLE(RESULT_SCAN('01a1b2c3-d4e5-f6g7-h8i9-0j1k2l3m4n5o'))
WHERE total_sales > 10000
ORDER BY total_sales DESC;
```

This approach avoids rerunning the aggregation and allows you to perform additional analysis efficiently.

## Summary

`RESULT_SCAN` is a powerful Snowflake feature for reusing query results, optimizing performance, and simplifying workflows. By leveraging query IDs or `LAST_QUERY_ID()`, you can treat previous results as virtual tables, apply filters, and integrate them into ELT or analytical processes without recomputation.

# COPY INTO Command — Deep Dive

---

## Important Attributes & Options

### 1. FILE_FORMAT
- Specifies the format of the data files being loaded
- **Two ways to specify:**
  - `FORMAT_NAME = 'my_format'` — reference a named file format object
  - `TYPE = CSV | JSON | AVRO | ORC | PARQUET | XML` — specify inline with options
- Common sub-options for CSV: `FIELD_DELIMITER`, `SKIP_HEADER`, `COMPRESSION`, `NULL_IF`, `ERROR_ON_COLUMN_COUNT_MISMATCH`

```sql
file_format = (TYPE = CSV FIELD_DELIMITER=',' SKIP_HEADER=1)
-- OR
file_format = (FORMAT_NAME = 'my_csv_format')
```

---

### 2. PATTERN
- A **regex pattern** to filter which files to load from the stage
- Useful when the stage contains many files and you only want specific ones

```sql
PATTERN = '.*Order.*'       -- loads files containing 'Order' in their name
PATTERN = '.*[.]csv[.]gz'  -- loads compressed CSV files only
```

---

### 3. FILES
- Specifies an **explicit list** of file names to load (max 1000 files)
- Generally the **fastest** option for file selection

```sql
FILES = ('file1.csv', 'file2.csv', 'file3.csv')
```

---

### 4. ON_ERROR
- Defines what happens when an error is encountered during loading

| Value | Behavior |
|-------|----------|
| `ABORT_STATEMENT` | **(Default for bulk)** Stop the entire load on first error |
| `CONTINUE` | Skip error rows, continue loading rest of file |
| `SKIP_FILE` | **(Default for Snowpipe)** Skip the entire file if any error found |
| `SKIP_FILE_n` | Skip file if number of errors >= n |
| `'SKIP_FILE_n%'` | Skip file if error percentage exceeds n% |

---

### 5. FORCE
- `FORCE = TRUE` — Reload files **regardless** of whether they were loaded before
- **Warning:** This can cause **duplicate data** in your table
- Default: `FALSE`

---

### 6. PURGE
- `PURGE = TRUE` — **Automatically delete** files from the stage after successful load
- Default: `FALSE`

---

### 7. VALIDATION_MODE
- Validates data from files **without actually loading** them
- Useful for testing before committing to a real load

| Value | Behavior |
|-------|----------|
| `RETURN_n_ROWS` | Validates first n rows, shows data as it would appear; fails at the first error encountered; |
| `RETURN_ERRORS` | Returns all errors across all files |
| `RETURN_ALL_ERRORS` | Returns all errors including from partially loaded files |

```sql
COPY INTO mytable VALIDATION_MODE = 'RETURN_ERRORS';
```

---

### 8. MATCH_BY_COLUMN_NAME
- Loads semi-structured data (JSON, Parquet, Avro, ORC, CSV) into table columns by **matching column names** instead of position
- Values: `CASE_SENSITIVE`, `CASE_INSENSITIVE`, `NONE` (default)
- Column order in file doesn't need to match table order

```sql
MATCH_BY_COLUMN_NAME = 'CASE_INSENSITIVE'
```

---

## Loading from Internal Stage vs External Stage

| Aspect | Internal Stage | External Stage |
|--------|---------------|----------------|
| Location | Stored within Snowflake (managed by Snowflake) | External cloud storage (S3, GCS, Azure) |
| File Upload | Use `PUT` command to upload files | Files already exist in cloud storage |
| Credentials | Not needed (Snowflake manages) | May need credentials or storage integration |
| Use Case | Local files, no external cloud access | Data already in cloud storage |
| Cost | No data transfer if same region | Possible data transfer costs |
| Stage Types | User stage (`@~`), Table stage (`@%table`), Named (`@stage`) | Named external stage (`@ext_stage`) |

---

## How Snowflake Tracks Loaded Files

Snowflake maintains **load metadata** for each table, which includes:
- **File name** (full path within the stage)
- **File size**
- **ETag** (a checksum/content hash of the file)
- Number of rows parsed
- Timestamp of last load
- Error information

> This metadata **expires after 64 days**. After that, the load status becomes "unknown" for old files.

---

## What Happens If the Same File Arrives Twice?

- By default, Snowflake **skips** files that were already loaded successfully (prevents duplicates)
- To reload, you must either:
  - Use `FORCE = TRUE` (reloads everything, risk of duplicates)
  - **Modify the file** and re-stage it (generates a new checksum/ETag)

---

## How Does Snowflake Know If a File Is New?

Snowflake checks **multiple attributes**, NOT just the file name:

- **File name** (path within the stage)
- **File size**
- **ETag / checksum** (content-based hash)

> If you re-stage the **same file with the same content** (same name + same checksum), Snowflake will **skip it** because it was already loaded.
>
> If you modify the file content and re-stage it, it gets a **new checksum** and Snowflake treats it as a new file.

**Key point:** It's not just the file name — Snowflake uses the combination of file path and content checksum to determine uniqueness.

---

## COPY INTO \<table\> vs COPY INTO \<location\>

| Aspect | COPY INTO \<table\> | COPY INTO \<location\> |
|--------|---------------------|------------------------|
| **Direction** | **Loading** data INTO Snowflake | **Unloading** data OUT of Snowflake |
| **Source** | Stage (internal/external) or cloud storage | Table or query result |
| **Destination** | Snowflake table | Stage or cloud storage location |
| **Purpose** | Ingest/import data | Export/extract data |
| **File Formats** | CSV, JSON, Avro, ORC, Parquet, XML | CSV, JSON, Parquet |
| **Example** | `COPY INTO my_table FROM @stage` | `COPY INTO @stage FROM my_table` |

```sql
-- Loading INTO a table
COPY INTO my_table FROM @my_stage;

-- Unloading FROM a table to a location
COPY INTO @my_stage FROM my_table;
```

# FILE FORMAT Object in Snowflake

---

## What is a File Format Object?
- Instead of specifying file format options **inline** in every COPY command, you can create a **reusable file format object**
- It's a **database object** (like a stage or table) that stores format properties
- **Best practice:** Create and reuse file format objects instead of repeating format options in every COPY command

---

## Creating a File Format Object

```sql
-- Create a schema to organize file formats (best practice)
CREATE OR REPLACE SCHEMA MANAGE_DB.file_formats;

-- Create a file format object (defaults to CSV with comma delimiter)
CREATE OR REPLACE FILE FORMAT MANAGE_DB.file_formats.my_file_format;
```

---

## Viewing Properties

```sql
DESC FILE FORMAT MANAGE_DB.file_formats.my_file_format;
```
> Shows ALL properties: type, field_delimiter, skip_header, compression, null_if, etc.

---

## Altering a File Format

```sql
ALTER FILE FORMAT MANAGE_DB.file_formats.my_file_format
    SET SKIP_HEADER = 1;
```

> **Important:** You **CANNOT change the TYPE** (e.g., CSV to JSON) using ALTER.
> Different types have different format options, so you must **recreate** the object using `CREATE OR REPLACE`.

---

## Using File Format in COPY Command

```sql
COPY INTO my_table
    FROM @my_stage
    file_format = (FORMAT_NAME = 'MANAGE_DB.file_formats.my_file_format');
```

---

## Overriding Properties in a Single COPY Command

You can use a file format object but **override specific properties** for just that one command:

```sql
COPY INTO my_table
    FROM @my_stage
    file_format = (FORMAT_NAME = 'MANAGE_DB.file_formats.my_file_format' SKIP_HEADER = 1);
```
> This does NOT alter the file format object — it only applies the override for this single COPY command.

---

## Creating File Format with Custom Properties

```sql
CREATE OR REPLACE FILE FORMAT MANAGE_DB.file_formats.my_file_format
    TYPE = JSON
    TIME_FORMAT = AUTO;
```

---

## Key Rules & Gotchas

- **Default type is CSV** — if you don't specify TYPE, it defaults to CSV
- **Cannot ALTER the TYPE** — CSV and JSON have different properties, so changing type requires recreating the object
- **Naming is case-insensitive** — Snowflake stores names in UPPERCASE by default

---

## File Format Properties in Stage Objects

- Stage objects **also contain** file format properties (type, field_delimiter, skip_header, etc.)
- Almost all file format properties are **available in both** stage and file format objects
- **Best practice:** Keep file format properties in a **separate file format object** rather than defining them in the stage
- When you specify a file format in COPY, it **overrides** the stage's file format properties

---

## Priority / Precedence Order

```
COPY command inline options  →  overrides  →  File Format Object  →  overrides  →  Stage Object properties
```

> The COPY command's inline options have the **highest priority**, followed by the file format object, followed by the stage defaults.

---

## Summary: Where Can File Format Be Defined?

| Location | Description | Reusable? |
|----------|-------------|----------|
| **Inline in COPY** | Specified directly in each COPY command | No |
| **File Format Object** | Separate database object | Yes |
| **Stage Object** | Properties stored within the stage | Tied to stage |

# Query ID in Snowflake

---

## What is a Query ID?
- Every query executed in Snowflake gets a **unique identifier** (UUID format)
- Example: `01b71944-0001-b181-0000-0129032279f6`
- Snowflake **persists query results for 24 hours** — you can access them using the query ID

---

## LAST_QUERY_ID() Function

Returns the query ID of a query in the **current session**.

```sql
-- Get the most recent query's ID
SELECT LAST_QUERY_ID();

-- Get the first query run in this session
SELECT LAST_QUERY_ID(1);

-- Get the second most recent query
SELECT LAST_QUERY_ID(-2);
```

| Argument | Meaning |
|----------|---------|
| `LAST_QUERY_ID()` or `(-1)` | Most recent query |
| `LAST_QUERY_ID(1)` | First query of the session |
| `LAST_QUERY_ID(-2)` | Second most recent query |

---

## RESULT_SCAN() — Run SELECT on Query Results

Converts the result of a previous query into a **table-like format** you can query with SELECT.

```sql
-- Query the result of the most recent query
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

-- Query using a specific query ID
SELECT * FROM TABLE(RESULT_SCAN('01b71944-0001-b181-0000-0129032279f6'));

-- Filter results from the previous query
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
    WHERE "column_name" = 'some_value';
```

---

## Practical Use Cases

### Query results of SHOW / DESCRIBE commands
```sql
SHOW TABLES;

SELECT "database_name", "schema_name", "name" AS "table_name"
    FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
    WHERE "rows" = 0;
```
> **Note:** Column names from SHOW/DESC commands are lowercase — use **double quotes** to reference them.

### Process stored procedure output
```sql
CALL my_procedure();

SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
```

---

## DESCRIBE RESULT — View Column Metadata of a Query

```sql
-- Describe columns of the most recent query's result
DESC RESULT LAST_QUERY_ID();

-- Describe columns using a specific query ID
DESC RESULT '01b71944-0001-b181-0000-0129032279f6';
```
> Returns column names, data types, nullable, default values, etc.

---

## Key Rules & Limitations

- Query results are stored for **24 hours only** (not adjustable)
- Only **the user who ran the query** can access its results via RESULT_SCAN
- RESULT_SCAN does **NOT guarantee row order** — use ORDER BY if needed
- Works across sessions (you can use a query ID from a past session within 24 hrs)
- `LAST_QUERY_ID()` only works within the **current session**

---

## Summary Table

| Function / Command | Purpose |
|---|---|
| `LAST_QUERY_ID()` | Get the query ID of recent queries in current session |
| `RESULT_SCAN(query_id)` | SELECT from a previous query's result set |
| `DESC RESULT query_id` | View column metadata of a query result |

## Query History Table Function in Snowflake

Snowflake provides several ways to access query history, allowing you to review past queries, monitor performance, and retrieve query IDs for use with functions like `RESULT_SCAN`.

---

### 1. Query History Table Function

```sql
SELECT *
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
ORDER BY START_TIME DESC;
```

Filter by user:

```sql
SELECT QUERY_ID, QUERY_TEXT, USER_NAME, START_TIME, END_TIME, EXECUTION_STATUS
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
WHERE USER_NAME = CURRENT_USER()
ORDER BY START_TIME DESC;
```

---

### 2. Last 7 Days Query History (Account Usage)

```sql
SELECT QUERY_ID, USER_NAME, QUERY_TEXT, START_TIME, END_TIME, EXECUTION_STATUS
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
ORDER BY START_TIME DESC;
```

Find a specific query:

```sql
SELECT *
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_TEXT ILIKE '%customer%'
ORDER BY START_TIME DESC;
```

---

### 3. Get Recently Executed Queries

```sql
SHOW QUERIES;
```

Or:

```sql
SHOW QUERIES LIMIT 100;
```

---

### 4. Get Details Using Query ID

```sql
SELECT *
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION())
WHERE QUERY_ID = '01b12345-0600-1234-0000-abcdef123456';
```

---

### 5. Query History for Current Session

```sql
SELECT *
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION())
ORDER BY START_TIME DESC;
```

---

### Useful Columns

| Column | Description |
|--------|-------------|
| `QUERY_ID` | Unique query identifier |
| `QUERY_TEXT` | SQL statement executed |
| `USER_NAME` | User who ran the query |
| `START_TIME` | When the query started |
| `END_TIME` | When the query finished |
| `EXECUTION_STATUS` | Success, failed, etc. |
| `ROWS_PRODUCED` | Number of rows returned |
| `WAREHOUSE_NAME` | Warehouse used |
| `TOTAL_ELAPSED_TIME` | Total time in milliseconds |

---

### Quick History Command (Most Common)

If you're looking for a Snowflake equivalent of a Linux/SQL Server **history command**, the most common query is:

```sql
SELECT QUERY_ID, QUERY_TEXT, START_TIME
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
ORDER BY START_TIME DESC;
```

## RESULT_SCAN in Snowflake

**RESULT_SCAN** allows you to retrieve the results of a previously executed query in Snowflake as a virtual table, enabling further querying without rerunning the original query.

---

### Overview

The `RESULT_SCAN` function in Snowflake is a **table function** that returns the result set of a previous query as if it were a table. This is particularly useful for:

- Inspecting or validating results from prior queries during debugging
- Processing outputs from `SHOW` or `DESCRIBE` commands
- Accessing results from queries on metadata or account usage
- Using results from stored procedures that return tabular data

`RESULT_SCAN` can reference queries from the current session or other sessions, as long as the query was executed within the **last 24 hours**.

---

### Syntax and Usage

The basic syntax is:

```sql
SELECT *
FROM TABLE(RESULT_SCAN('<query_id>'));
```

Where `<query_id>` is the unique identifier of the query whose results you want to retrieve. You can obtain the query ID from:

- Snowflake Web UI under **Monitoring > Query History**
- `QUERY_HISTORY` table function
- `LAST_QUERY_ID()` function for the most recent query in the current session

For example, retrieving the last query's results:

```sql
SELECT *
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
```

You can also apply additional SQL clauses like `WHERE` or `ORDER BY` to filter or sort the results differently from the original query.

---

### Practical Considerations

- **Duplicate column names** in the original query are automatically renamed with suffixes (`_1`, `_2`, etc.) to ensure uniqueness.
- **Performance**: `RESULT_SCAN` returns cached results, which is faster than rerunning the query, but large result sets may be slower to process than querying a table directly.
- **Integration**: It is useful in ELT workflows, allowing intermediate results to be reused without creating temporary tables, reducing compute costs.

---

### Example

Suppose you run a query to calculate total sales per region:

```sql
SELECT region, SUM(sales) AS total_sales
FROM sales_data
GROUP BY region;
```

After capturing the query ID (e.g., `01a1b2c3-d4e5-f6g7-h8i9-0j1k2l3m4n5o`), you can retrieve and further filter the results:

```sql
SELECT *
FROM TABLE(RESULT_SCAN('01a1b2c3-d4e5-f6g7-h8i9-0j1k2l3m4n5o'))
WHERE total_sales > 10000
ORDER BY total_sales DESC;
```

This approach avoids rerunning the aggregation and allows you to perform additional analysis efficiently.

---

### Summary

`RESULT_SCAN` is a powerful Snowflake feature for reusing query results, optimizing performance, and simplifying workflows. By leveraging query IDs or `LAST_QUERY_ID()`, you can treat previous results as virtual tables, apply filters, and integrate them into ELT or analytical processes without recomputation.